LAYER 2 : REASONING ENGINE
This is where the agent starts reasoning like a human.
The notebook covers Decision intelligence across:

compatibility reasoning, contextual reasoning, rating prediction, review planning, recommendation reasoning.


PHASE 1 - Persona-Item Compatibility Engine
We’ll compute compatibility using:

- dominant values	(ambience vs affordability).
- cuisine preferences	(likes Japanese food).
- communication identity	(expressive vs analytical).
- Nigerian identity	(soft-life vs practical).
- emotional style	(optimistic vs critical).

Then combine them into one interpretable compatibility score.

PHASE 1 ROADMAP

We’ll implement:

Step	Goal
1	- Build restaurant feature profiles
2	- Build cuisine preference matching
3	- Build value compatibility
4	- Build Nigerian identity matching
5	- Combine into unified score
6	- Generate reasoning explanations

In [101]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

In [102]:
import pandas as pd
import numpy as np

In [ ]:
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()

genai.configure(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [9]:
model = genai.GenerativeModel(
    "models/gemini-flash-latest"
)

In [10]:
response = model.generate_content(
    "Say hello"
)

print(response.text)

Hello! How can I help you today?


In [103]:
from src.generation.prompt_builder import (
    build_review_prompt
)

from src.generation.review_generator import (
    generate_review
)

In [11]:
# STEP 1 — LOAD PERSONA ENGINE OUTPUT
persona_df = pd.read_csv(
    "../data/processed/persona_profiles.csv"
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,time_efficiency_y,time_efficiency_y.1,naija_narrative,hyperbole_narrative,kinship,oral_connectors,nigerian_identity,persona_summary,recommendation_tendency,cognitive_profile
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic,"{'positivity': 'low', 'verbosity': 'high', 'em...",...,0.222222,0.333333,0.0,0.0,0.000000,0.000000,pidgin_staples,Archetype: Harsh Critic\n\n Communication S...,prioritizes authentic and high-quality meals,"{'archetype': 'Harsh Critic', 'communication_s..."
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.227273,0.318182,0.0,0.0,0.000000,0.045455,pidgin_staples,Archetype: Emotional Storyteller\n\n Commun...,"expects courteous, attentive, and reliable staff","{'archetype': 'Emotional Storyteller', 'commun..."
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.250000,0.458333,0.0,0.0,0.041667,0.000000,pidgin_staples,Archetype: Emotional Storyteller\n\n Commun...,prioritizes authentic and high-quality meals,"{'archetype': 'Emotional Storyteller', 'commun..."
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist,"{'positivity': 'high', 'verbosity': 'moderate'...",...,0.066667,0.066667,0.0,0.0,0.000000,0.000000,pidgin_staples,Archetype: Warm Optimist\n\n Communication ...,balanced preferences,"{'archetype': 'Warm Optimist', 'communication_..."
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate...",...,0.136364,0.136364,0.0,0.0,0.000000,0.000000,pidgin_staples,Archetype: Reactive Reviewer\n\n Communicat...,"prefers easy access, fast delivery, and hassle...","{'archetype': 'Reactive Reviewer', 'communicat..."


In [12]:
# STEP 2 — LOAD YELP BUSINESS DATA
businesses = pd.read_json(
    "../data/raw/yelp_academic_dataset_business.json",
    lines=True
)

businesses.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."


In [13]:
# STEP 3 — LOAD CURATED REVIEWS (IF SAVED)

curated_reviews = pd.read_csv(
    "../data/processed/curated_reviews.csv"
)

curated_reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,sentiment,...,persuasion,blame_criticism,humour_sarcasm,cultural_references,practical_survival,time_efficiency.1,naija_narrative,hyperbole_narrative,kinship,oral_connectors
0,6Vf1MkxDPrTcnuKYUldjFw,-EX1hrPRBqNkVavtMllTCA,agK5cXwnBQozM2M-5kLvzw,1,8,2,2,"We had food from this ""restaurant"" delivered t...",2011-12-15 12:02:27,-0.300000,...,0,0,0,0,0,0,0,0,0,0
1,Aninptga9OOZsmi5gFNAjQ,-EX1hrPRBqNkVavtMllTCA,XnVmNQdmyhdCC90FRY9ZJQ,5,0,0,0,Had our Christmas party here this year. The s...,2011-12-16 23:14:10,0.540179,...,0,0,0,0,0,0,0,0,0,0
2,EiPqGnP5SRapBaOkvZJAug,-EX1hrPRBqNkVavtMllTCA,GBTPC53ZrG1ZBY3DT8Mbcw,4,1,0,1,"Went here on a lark, basically. I had enough ...",2012-02-04 17:10:19,0.298437,...,0,0,0,0,0,0,0,0,0,0
3,pL3LWEwcqaTLaa7c4o860A,-EX1hrPRBqNkVavtMllTCA,-9yzQQ0d_rcOD2CzdTNO_Q,5,1,1,2,"This is one of the ""retro"" McDonald's and it's...",2012-02-07 18:27:15,0.276420,...,0,0,0,0,0,0,0,0,0,0
4,46Z1SVg6VJygnN3O95cFsw,-EX1hrPRBqNkVavtMllTCA,FEFi0AmjHzgSceeLYW3Glw,2,0,0,0,This location is inconsistent. Sometimes it b...,2012-02-07 18:49:57,-0.150000,...,0,0,0,0,0,1,0,0,0,0


In [14]:
# STEP 3 — CREATE BUSINESS FEATURE TABLE

business_profiles = businesses[
    [
        "business_id",
        "name",
        "categories",
        "stars",
        "review_count"
    ]
].copy()

business_profiles.head()

,business_id,name,categories,stars,review_count
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","Doctors, Traditional Chinese Medicine, Naturop...",5.0,7
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,"Shipping Centers, Local Services, Notaries, Ma...",3.0,15
2,tUFrWirKiKi_TAnsVWINQQ,Target,"Department Stores, Shopping, Fashion, Home & G...",3.5,22
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",4.0,80
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,"Brewpubs, Breweries, Food",4.5,13


In [15]:
business_profiles = businesses[
    [
        "business_id",
        "name",
        "categories",
        "stars",
        "review_count"
    ]
].copy()

In [16]:
# STEP 4 — CLEAN CATEGORY TEXT

business_profiles["categories"] = (
    business_profiles["categories"]
    .fillna("")
)

In [17]:
# STEP 5 - CREATE BUSINESS ATTRIBUTES

def extract_business_traits(category_text):

    text = category_text.lower()

    traits = {

        "ambience": 0,
        "luxury": 0,
        "convenience": 0,
        "social": 0,
        "casual": 0,
        "food_focus": 0,
        "shopping": 0,
        "time_efficiency": 0,
        "durability": 0
    }

    ambience_words = [
        "lounges", "cafe", "cafes", "rooftop", "wine", "cocktail", "ambience", "atmosphere", "decor", "vibes", 
        "music", "aesthetic", "cozy", "lighting", "cleanliness", "noise level", "comfortable seating", 
        "romantic", "family-friendly", "decoration choke", "overcrowded", "spacious", "intimate", "loud", 
        "quiet", "coffee", "tea", "desserts", "brunch", "bakery", "garden"
    
    ]

    luxury_words = [
        "fine dining", "steakhouse", "upscale", "premium", "luxury", "club", "sports bar", "golf course",
        "upscale", "fancy", "high-end", "exclusive", "luxurious", "opulent", 
        "lavish", "posh", "sophisticated", "elegant", "glamorous", "hotel", 
        "resort", "spa", "gourmet", "Michelin", "sommelier", "champagne", "caviar"
    ]

    convenience_words = [
        "fast", "quick", "parking", "location", "accessible", "easy",  "waiting time"
        "near me", "home delivery", "takeaway", "drive-thru", "curbside pickup", "self-service"
    ]

    social_words = [
        "bars", "nightlife", "music", "clubs", "friends", "family", "date", "group", "celebration", "birthday", "hangout"
        "social gathering", "romantic dinner", "family outing", "friend meetup", "special occasion"
        "anniversary", "reunion", "casual hangout", "work event", "holiday celebration"
        "new spot to try", "place to see and be seen", "vibe for socializing", "perfect for groups", "intimate setting",
        "sports bars", "karaoke", "beer", "pub"
    ]

    casual_words = [
        "fast food", "pizza", "burgers", "sandwiches", "food trucks", "takeout"
    ]

    food_words = [
        "restaurants", "seafood", "sushi", "bbq", "ramen", "mexican", "italian", "thai", 
        "korean", "indian", "chinese", "vegetarian", "vegan", "gluten-free", "desserts", "brunch", "bakery"
    ]

    shopping_words = [
        "groceries", "home", "kitchen", "electronics"
    ]

    time_words = [
        "wait time", "delay", "fast delivery", "slow", "African time",
        "hours", "minutes", "late", "early", "prompt", "wasted my time",
        "traffic", "Lagos traffic", "delivered on time", "arrived late", 
        "arrived early", "on schedule", "behind schedule", "ahead of schedule", "fast"
    ]

    durability_words = [
        "original", "fake", "counterfeit", "rugged", "last long", "strong",
        "weak", "fragile", "repair", "spare parts", "generator", "battery life",
        "heat up", "spoilt", "still working", "tested and trusted", "tokunbo", "new", "used"
    ]

    for word in ambience_words:
        if word in text:
            traits["ambience"] += 1

    for word in luxury_words:
        if word in text:
            traits["luxury"] += 1

    for word in social_words:
        if word in text:
            traits["social"] += 1

    for word in casual_words:
        if word in text:
            traits["casual"] += 1

    for word in food_words:
        if word in text:
            traits["food_focus"] += 1

    for word in shopping_words:
        if word in text:
            traits["shopping"] += 1
    
    for word in time_words:
        if word in text:
            traits["time_efficiency"] += 1

    for word in convenience_words:
        if word in text:
            traits["convenience"] += 1

    for word in durability_words:
        if word in text:
            traits["durability"] += 1

    return traits

In [18]:
# STEP 6 — APPLY BUSINESS TRAIT EXTRACTION

business_profiles["traits"] = (
    business_profiles["categories"]
    .fillna("")
    .apply(extract_business_traits)
)

In [19]:
# STEP 7 — EXPAND BUSINESS TRAITS

business_traits_df = pd.json_normalize(
    business_profiles["traits"]
)

business_traits_df.head()

,ambience,luxury,convenience,social,casual,food_focus,shopping,time_efficiency,durability
0,0,0,0,0,0,1,0,0,0
1,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,2,0,0
3,2,0,0,0,0,1,0,0,0
4,0,0,0,1,0,0,0,0,0


In [20]:
# STEP 8 — MERGE TRAITS INTO BUSINESS TABLE
# Transforming raw business metadata into interpretable environmental signals.

business_profiles = pd.concat(
    [business_profiles, business_traits_df],
    axis=1
)

business_profiles.head(10)

,business_id,name,categories,stars,review_count,traits,ambience,luxury,convenience,social,casual,food_focus,shopping,time_efficiency,durability
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","Doctors, Traditional Chinese Medicine, Naturop...",5.0,7,"{'ambience': 0, 'luxury': 0, 'convenience': 0,...",0,0,0,0,0,1,0,0,0
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,"Shipping Centers, Local Services, Notaries, Ma...",3.0,15,"{'ambience': 0, 'luxury': 0, 'convenience': 0,...",0,0,0,0,0,0,0,0,0
2,tUFrWirKiKi_TAnsVWINQQ,Target,"Department Stores, Shopping, Fashion, Home & G...",3.5,22,"{'ambience': 1, 'luxury': 0, 'convenience': 0,...",1,0,0,0,0,0,2,0,0
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",4.0,80,"{'ambience': 2, 'luxury': 0, 'convenience': 0,...",2,0,0,0,0,1,0,0,0
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,"Brewpubs, Breweries, Food",4.5,13,"{'ambience': 0, 'luxury': 0, 'convenience': 0,...",0,0,0,1,0,0,0,0,0
5,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",2.0,6,"{'ambience': 0, 'luxury': 0, 'convenience': 1,...",0,0,1,0,3,1,0,1,0
6,n_0UpQx1hsNbnPUSlodU8w,Famous Footwear,"Sporting Goods, Fashion, Shoe Stores, Shopping...",2.5,13,"{'ambience': 0, 'luxury': 0, 'convenience': 0,...",0,0,0,0,0,0,0,0,0
7,qkRM_2X51Yqxk3btlwAQIg,Temple Beth-El,"Synagogues, Religious Organizations",3.5,5,"{'ambience': 0, 'luxury': 0, 'convenience': 0,...",0,0,0,0,0,0,0,0,0
8,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,"Pubs, Restaurants, Italian, Bars, American (Tr...",3.0,19,"{'ambience': 0, 'luxury': 0, 'convenience': 0,...",0,0,0,3,0,2,0,0,0
9,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,...",1.5,10,"{'ambience': 0, 'luxury': 0, 'convenience': 1,...",0,0,1,0,2,1,0,1,0


In [21]:
# STEP 9 — CREATE VALUE COMPATIBILITY FUNCTION
# We now compare user values VS restaurant traits.

def value_compatibility(persona_row, business_row):

    score = 0

    dominant_value = persona_row["dominant_value"]

    if dominant_value == "ambience":
        score += business_row["ambience"]

    elif dominant_value == "social_experience":
        score += business_row["social"]

    elif dominant_value == "food_quality":
        score += business_row["food_focus"]

    elif dominant_value == "affordability":
        score += business_row["casual"]

    elif dominant_value == "time":
            score += business_row["time_efficiency"]

    elif dominant_value == "convenience":
            score += business_row["convenience"]

    elif dominant_value == "durability":
            score += business_row["durability"]

    elif dominant_value == "shopping":
            score += business_row["shopping"]

    return score

In [22]:
# STEP 10 — CREATE NIGERIAN COMPATIBILITY FUNCTION
# cultural alignment.
# We can create a function that assesses how well a business aligns with the specific cultural identities we've identified in our Nigerian personas. 
# This function will take into account the unique traits and preferences associated with each Nigerian identity and compare them against the features of the businesses.


def nigerian_compatibility(persona_row, business_row):

    identity = persona_row["nigerian_identity"]

    score = 0

    if identity == "soft_life_explorer":
        score += (
            business_row["luxury"]
            + business_row["ambience"]
        )

    elif identity == "social_enjoyment":
        score += business_row["social"]

    elif identity == "practical_survivor":
        score += business_row["casual"]

    elif identity == "time_efficiency":
        score += business_row["time_efficiency"]

    elif identity == "pidgin_staples":
        score += business_row["casual"] + business_row["social"]

        
    return score

In [23]:
# STEP 11 — CREATE COMPATIBILITY ENGINE

def compute_compatibility(
    persona_row,
    business_row
):

    value_score = value_compatibility(
        persona_row,
        business_row
    )

    nigerian_score = nigerian_compatibility(
        persona_row,
        business_row
    )

    final_score = (
        value_score * 0.7
        + nigerian_score * 0.3
    )

    return {
        "value_score": value_score,
        "nigerian_score": nigerian_score,
        "final_score": final_score
    }

In [24]:
# Remove duplicate columns

business_profiles = business_profiles.loc[
    :,
    ~business_profiles.columns.duplicated()
]

In [25]:
# STEP 12 — TEST COMPATIBILITY ENGINE

# Now we test with one persona + one restaurant

sample_persona = persona_df.iloc[7]

sample_business = business_profiles.iloc[81]

compute_compatibility(
    sample_persona,
    sample_business
)

{'value_score': np.int64(0),
 'nigerian_score': np.int64(1),
 'final_score': np.float64(0.3)}

In [26]:
# 12a: Get persona details
# This helps you retrieve persona information to test for compatibility with a business

persona_df[
    [
        "user_id",
        "archetype",
        "dominant_value",
        "nigerian_identity",
        "communication_style"
    ]
].head(40)

,user_id,archetype,dominant_value,nigerian_identity,communication_style
0,-EX1hrPRBqNkVavtMllTCA,Harsh Critic,food_quality,pidgin_staples,balanced
1,-M7fUg7FrdGctKr5f_eMUQ,Emotional Storyteller,service_quality,pidgin_staples,balanced
2,-WM58wLjtlHlR91xVfM1FQ,Emotional Storyteller,food_quality,pidgin_staples,balanced
3,-qTtg1D3RidRa4cTB-ftwg,Warm Optimist,positive_exaggeration,pidgin_staples,balanced
4,02H49g16MdRoZKoX6IEoFA,Reactive Reviewer,convenience,pidgin_staples,balanced
5,0650daOKAuufqymyOBe3cA,Warm Optimist,service_quality,pidgin_staples,balanced
6,06Yz-YYYa1U9PN37b6UniA,Emotional Storyteller,food_quality,pidgin_staples,balanced
7,0Tsu6-uhw_w9Z3W0Xlnazg,Harsh Critic,affordability,pidgin_staples,balanced
8,0xJwTzZuWOac7ufeTioeig,Emotional Storyteller,food_quality,pidgin_staples,balanced
9,175DuOm7IPiEy5f1KG9esA,Harsh Critic,food_quality,pidgin_staples,balanced


In [27]:
# Get specific persona user ID

target_user_id = "1kdfj_PaRk8i870ghdIvXg"

In [28]:
target_persona = persona_df[
    persona_df["user_id"] == target_user_id
]

target_persona.T

,16
user_id,1kdfj_PaRk8i870ghdIvXg
avg_rating,3.967742
rating_variance,0.948116
review_count,31
avg_review_length,446.83871
...,...
oral_connectors,0.0
nigerian_identity,pidgin_staples
persona_summary,Archetype: Emotional Storyteller\n\n Commun...
recommendation_tendency,"expects courteous, attentive, and reliable staff"


In [29]:
# Get selected persona user review to get insight into their preference
user_reviews = curated_reviews[
    curated_reviews["user_id"] == target_user_id
]

user_reviews[
    [
        "stars",
        "categories",
        "text"
    ]
].head(5)

,stars,categories,text
413,3,"Basque, Tapas Bars, Spanish, Restaurants",Stopped in for dinner. The atmosphere was mod...
414,3,"Middle Eastern, Greek, Restaurants",This is actually my second time coming here. ...
415,3,"Mobile Phones, Electronics, Fashion, Shopping,...",This Wal-Mart is always overcrowded!!! My Wal...
416,2,"Soul Food, Restaurants, Caribbean, Latin American",I was very disappointed!!! I make a mean curr...
417,3,"Southern, Comfort Food, Caterers, Breakfast & ...",My Cracker Barrel experience was nice. My hus...


In [30]:
#Business Information

candidate_businesses = business_profiles[
    business_profiles["categories"]
    .str.contains(
        "Restaurants|Spanish|Italian|Steakhouses|Chicken Wings",
        case=False,
        na=False
    )
]

candidate_businesses[
    [
        "business_id",
        "name",
        "categories",
        "ambience",
        "luxury",
        "social"
    ]
].head(20)

,business_id,name,categories,ambience,luxury,social
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",2,0,0
5,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",0,0,0
8,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,"Pubs, Restaurants, Italian, Bars, American (Tr...",0,0,3
9,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,...",0,0,0
11,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,"Vietnamese, Food, Restaurants, Food Trucks",0,0,0
12,il_Ro8jwPlHresjw9EGmBg,Denny's,"American (Traditional), Restaurants, Diners, B...",1,0,0
14,0bPLkL0QhhPO5kt1_EXmNQ,Zio's Italian Market,"Food, Delis, Italian, Bakeries, Restaurants",0,0,0
15,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,"Sushi Bars, Restaurants, Japanese",0,0,1
19,ROeacJQwBeh05Rqg7F6TCg,BAP,"Korean, Restaurants",0,0,0
20,WKMJwqnfZKsAae75RMP6jA,Roast Coffeehouse and Wine Bar,"Coffee & Tea, Food, Cafes, Bars, Wine Bars, Re...",5,0,2


In [31]:
business_profiles.iloc[35]

business_id                                   aPNXGTDkf-4bjhyMBQxqpQ
name                                                      Craft Hall
categories         Eatertainment, Arts & Entertainment, Brewpubs,...
stars                                                            3.5
review_count                                                      65
traits             {'ambience': 0, 'luxury': 0, 'convenience': 0,...
ambience                                                           0
luxury                                                             0
convenience                                                        0
social                                                             1
casual                                                             0
food_focus                                                         1
shopping                                                           0
time_efficiency                                                    0
durability                        

In [32]:
#Select Business ID
target_business = candidate_businesses.iloc[93]

In [33]:
compatibility = compute_compatibility(
    target_persona.iloc[0],
    target_business
)

compatibility

{'value_score': 0,
 'nigerian_score': np.int64(1),
 'final_score': np.float64(0.3)}

In [34]:
# STEP 13 — GENERATE HUMAN-READABLE REASONING

def explain_compatibility(
    persona_row,
    business_row,
    compatibility_result
):

    reasons = []

    if compatibility_result["value_score"] > 0:

        reasons.append(
            f"Matches the user's value for "
            f"{persona_row['dominant_value']}"
        )

    if compatibility_result["nigerian_score"] > 0:

        reasons.append(
            f"Aligns with the user's "
            f"Nigerian identity "
            f"({persona_row['nigerian_identity']})"
        )

    if len(reasons) == 0:

        reasons.append(
            "Limited behavioral alignment detected"
        )

    return reasons

In [35]:
# STEP 14 - TEST EXPLANATION ENGINE
compatibility = compute_compatibility(
    sample_persona,
    sample_business
)

explain_compatibility(
    sample_persona,
    sample_business,
    compatibility
)

["Aligns with the user's Nigerian identity (pidgin_staples)"]

PHASE 1 — Persona-Item Compatibility Engine is now Complete

The agent can now evaluate behavioral alignment, reason about compatibility, explain recommendations and model culturally-aware preference matching

PHASE 2: CONTEXT TAXONOMY

In [36]:
# STEP 15 — CREATE CONTEXT TAXONOMY
# modeling situational psychology.

context_taxonomy = {

    "weekday_quick_meal": {

        "prefers": [
            "casual",
            "affordability"
        ],

        "avoids": [
            "luxury"
        ],

        "mood": "busy",

        "group_size": "just_me",

        "economic_context": "budget_conscious"
    },

    "social_night_out": {

        "prefers": [
            "social",
            "ambience"
        ],

        "avoids": [],

        "mood": "energetic",

        "group_size": "small_group",

        "economic_context": "flexible_budget"
    },

    "celebration": {

        "prefers": [
            "luxury",
            "ambience"
        ],

        "avoids": [
            "casual"
        ],

        "mood": "celebratory",

        "group_size": "group",

        "economic_context": "splurging"
    },

    "comfort_food_mood": {

        "prefers": [
            "casual",
            "food_focus",
            "authentic_local"
        ],

        "avoids": [],

        "mood": "tired",

        "group_size": "just_me",

        "economic_context": "normal"
    },

    "work_cafe_session": {

        "prefers": [
            "ambience",
            "quiet"
        ],

        "avoids": [
            "social"
        ],

        "mood": "focused",

        "group_size": "just_me",

        "economic_context": "moderate"
    }
}

In [37]:
# CONTEXT DIMENSIONS
occasion_types = [

    "casual", "business_meeting", "formal", "celebration", "quick_bite", "takeaway", "delivery",
    "brunch", "after_work", "late_night"
]

group_sizes = [

    "just_me", "couple", "small_group", "large_group",
    "crowd"
]

moods = [

    "happy", "celebratory", "relaxed", "stressed", "tired", "hungry",
    "excited", "romantic", "chilled", "focused"
]

lagos_locations = [

    "Victoria_Island", "Lekki", "Ikoyi", "Yaba", "Surulere",
    "Ikeja", "Ajah", "Mainland", "Island"
]

economic_contexts = [

    "salary_day", "sapa_period", "splurging", "budget_conscious", "treat_yourself"
]

In [38]:
#STEP 16 — CREATE CONTEXT COMPATIBILITY FUNCTION
# Now we score business fit for current context.

def context_compatibility(
    business_row,
    context_name
):

    context = context_taxonomy[context_name]

    score = 0

    for preference in context["prefers"]:

        if preference in business_row:
            score += business_row[preference]

    for avoidance in context["avoids"]:

        if avoidance in business_row:
            score -= business_row[avoidance]

    return score

In [39]:
business_profiles.iloc[153]

business_id                                   aNtKyc2rr-uK5cqzY9TVQQ
name                                          Chipotle Mexican Grill
categories                           Mexican, Fast Food, Restaurants
stars                                                            3.0
review_count                                                      19
traits             {'ambience': 0, 'luxury': 0, 'convenience': 1,...
ambience                                                           0
luxury                                                             0
convenience                                                        1
social                                                             0
casual                                                             1
food_focus                                                         2
shopping                                                           0
time_efficiency                                                    1
durability                        

In [40]:
# STEP 17 — TEST CONTEXTUAL DIFFERENCES

sample_business = business_profiles.iloc[153]

for context_name in context_taxonomy.keys():

    score = context_compatibility(
        sample_business,
        context_name
    )

    print(context_name, "→", score)

weekday_quick_meal → 1
social_night_out → 0
celebration → -1
comfort_food_mood → 3
work_cafe_session → 0


The same restaurant now scores differently depending on context.

This is dynamic reasoning.

In [41]:
# STEP 18 — CREATE CONTEXT-AWARE COMPATIBILITY ENGINE
# Now combine persona compatibility and situational compatibility

def contextualized_compatibility(
    persona_row,
    business_row,
    context_name
):

    base_scores = compute_compatibility(
        persona_row,
        business_row
    )

    context_score = context_compatibility(
        business_row,
        context_name
    )

    final_score = (
        base_scores["final_score"] * 0.7
        + context_score * 0.3
    )

    return {

        "base_score":
            base_scores["final_score"],

        "context_score":
            context_score,

        "final_score":
            final_score
    }

In [42]:
# STEP 19 — TEST CONTEXTUALIZED REASONING
# Now we see how the same business can score differently for the same persona under different contexts.

sample_persona = persona_df.iloc[7]

sample_business = business_profiles.iloc[9]

contextualized_compatibility(
    sample_persona,
    sample_business,
    "celebration" # can be changed to other contexts like "social_night_out", "celebration", "work_cafe_session"
)

{'base_score': np.float64(2.0),
 'context_score': np.int64(-2),
 'final_score': np.float64(0.7999999999999999)}

In [43]:
contextualized_compatibility(
    sample_persona,
    sample_business,
    "weekday_quick_meal"
)

{'base_score': np.float64(2.0),
 'context_score': np.int64(2),
 'final_score': np.float64(2.0)}

The SAME user, in the same restaurant now produces different compatibility scores depending on situation.

This is akin to human-like reasoning.

In [44]:
# STEP 19 — GENERATE CONTEXTUAL EXPLANATIONS
# Now we can explain the reasoning behind the context-aware compatibility scores in a human-readable way.

def explain_contextual_reasoning(
    persona_row,
    business_row,
    context_name,
    result
):

    explanation = []

    explanation.append(
        f"Context: {context_name}"
    )

    if result["context_score"] > 0:

        explanation.append(
            "This venue aligns well with the current situation."
        )

    else:

        explanation.append(
            "This venue may not strongly fit the current situation."
        )

    explanation.extend(

        explain_compatibility(
            persona_row,
            business_row,
            compute_compatibility(
                persona_row,
                business_row
            )
        )
    )

    return explanation

In [45]:
# STEP 20 — TEST CONTEXTUAL EXPLANATION

result = contextualized_compatibility(
    sample_persona,
    sample_business,
    "celebration"
)

explain_contextual_reasoning(
    sample_persona,
    sample_business,
    "celebration",
    result
)

['Context: celebration',
 'This venue may not strongly fit the current situation.',
 "Matches the user's value for affordability",
 "Aligns with the user's Nigerian identity (pidgin_staples)"]

PHASE 2 — Contextual + Cultural Reasoning is now complete

The agent can now: reason contextually
- understand Nigerian cultural signals
- adapt recommendations culturally
- model situational behavior
- align personas with local behavioral styles

PHASE 3 — RATING PREDICTION ENGINE
Before humans write reviews they already internally decide, “How good or bad was this experience?”

That becomes the star rating. The written review is usually a justification of that score.

So our architecture becomes: 
persona + context + restaurant compatibility + emotional state = predicted rating = generated review

BUILD:
heuristic rating prediction, 
compatibility-driven scoring, 
personality-adjusted ratings, 
emotional drift effects, 
contextual rating modifiers.

In [46]:
# STEP 21 — CREATE RATING PREDICTION FUNCTION
# Finally, we can create a function that translates the compatibility scores into a predicted star rating (1-5) that the user might give to the business.
def base_rating_prediction(
    compatibility_score
):

    if compatibility_score >= 3:
        return 5

    elif compatibility_score >= 2:
        return 4

    elif compatibility_score >= 1:
        return 3

    elif compatibility_score >= 0:
        return 2

    else:
        return 1
    
# This creates : compatibility → satisfaction mapping.

In [47]:
# STEP 22 — ADD PERSONALITY RATING BIAS
# We can also add a bias factor based on the user's communication style or personality traits that we've identified in the persona profiles. 
# For example, some personas might be more likely to give higher ratings due to their optimistic nature, while others might be more critical. 
# This bias can be a simple adjustment to the predicted rating based on the persona's archetype or communication style.
# Harsh Critic → harsher ratings
# Warm Optimist → forgiving ratings

personality_rating_bias = {

    "Warm Optimist": 0.5,

    "Reactive Reviewer": 0,

    "Harsh Critic": -1,

    "Emotional Storyteller": 0.3,

    "Deep Experience Analyst": -0.3
}

In [48]:
# Step 23 — CREATE PERSONALITY-ADJUSTED RATING PREDICTION FUNCTION
# This function takes the base compatibility score, converts it to a rating, and then adjusts it based on the persona's archetype bias.

def personality_adjusted_rating(
    persona_row,
    compatibility_score
):

    base_rating = base_rating_prediction(
        compatibility_score
    )

    bias = personality_rating_bias[
        persona_row["archetype"]
    ]

    adjusted = base_rating + bias

    adjusted = round(adjusted)

    adjusted = max(1, min(5, adjusted))

    return adjusted

In [49]:
# STEP 24 — TEST PERSONALITY-ADJUSTED RATING PREDICTION 
# Now we can see how the same compatibility score might translate into different predicted ratings for different personas based on their archetype biases.

compatibility_score = 2

for archetype in personality_rating_bias.keys():

    persona_sample = persona_df[
        persona_df["archetype"] == archetype
    ].iloc[0]

    prediction = personality_adjusted_rating(
        persona_sample,
        compatibility_score
    )

    print(archetype, "→", prediction)

Warm Optimist → 4
Reactive Reviewer → 4
Harsh Critic → 3
Emotional Storyteller → 4
Deep Experience Analyst → 4


The SAME experience produces different ratings depending on personality. This is behavioral realism.

In [50]:
# STEP 25 — ADD CONTEXT RATING BIAS
# We can also add a bias based on the current context, as some contexts might lead to 
# higher or lower ratings on average. For example, people might be more forgiving in a celebratory context, 
# but more critical in a stressful or work-related context.

context_rating_bias = {

    "celebration": 0.5,

    "social_night_out": 0.3,

    "comfort_food_mood": 0.2,

    "weekday_quick_meal": -0.2,

    "work_cafe_session": -0.1
}

In [51]:
# Step 26 — CREATE CONTEXT-AWARE RATING PREDICTION FUNCTION
# This function combines the base compatibility score, the personality bias, and the context bias to produce a final predicted rating.

def contextual_rating_prediction(
    persona_row,
    compatibility_result,
    context_name
):

    compatibility_score = (
        compatibility_result["final_score"]
    )

    personality_rating = (
        personality_adjusted_rating(
            persona_row,
            compatibility_score
        )
    )

    context_bias = (
        context_rating_bias[context_name]
    )

    final_rating = (
        personality_rating + context_bias
    )

    final_rating = round(final_rating)

    final_rating = max(1, min(5, final_rating))

    return final_rating

In [52]:
# STEP 27 — TEST CONTEXTUAL RATING PREDICTION
# Now we can see how the same business might receive different predicted ratings from the same persona under different contexts due to the context bias.

sample_persona = persona_df.iloc[10]

sample_business = business_profiles.iloc[70]

compatibility_result = (
    contextualized_compatibility(
        sample_persona,
        sample_business,
        "celebration"
    )
)

contextual_rating_prediction(
    sample_persona,
    compatibility_result,
    "celebration"
)

2

In [53]:
contextual_rating_prediction(
    sample_persona,
    compatibility_result,
    "weekday_quick_meal"
)

1

In [54]:
# STEP 28 — EXPLAIN RATING PREDICTION
# Finally, we can create an explanation function that breaks down the reasoning behind the 
# predicted rating in a human-readable way, incorporating the persona's archetype, the context, and the compatibility scores.

def explain_rating_prediction(
    persona_row,
    context_name,
    predicted_rating
):

    explanation = []

    explanation.append(
        f"Predicted rating: {predicted_rating} stars"
    )

    explanation.append(
        f"User archetype: "
        f"{persona_row['archetype']}"
    )

    explanation.append(
        f"Context: {context_name}"
    )

    if predicted_rating >= 4:

        explanation.append(
            "Strong behavioral alignment detected."
        )

    elif predicted_rating == 3:

        explanation.append(
            "Moderate alignment with mixed signals."
        )

    else:

        explanation.append(
            "Weak alignment or dissatisfaction likely."
        )

    return explanation

In [55]:
# STEP 28 — TEST EXPLANATION OF RATING PREDICTION
# Now we can see how the explanation function breaks down the reasoning behind the predicted rating in a human-readable way.

predicted_rating = (
    contextual_rating_prediction(
        sample_persona,
        compatibility_result,
        "celebration"
    )
)

explain_rating_prediction(
    sample_persona,
    "celebration",
    predicted_rating
)

['Predicted rating: 2 stars',
 'User archetype: Emotional Storyteller',
 'Context: celebration',
 'Weak alignment or dissatisfaction likely.']

PHASE 3 — Rating Prediction Engine is now complete

The system can now:

- simulate satisfaction
- predict ratings behaviorally
- adjust scores contextually
- reflect personality biases
- explain predicted decisions

PHASE 4 — REVIEW PLANNING ENGINE
Humans experience an internal planning process to writinf reviews.
Before writing people decide:

- what mattered most
- what emotion dominated
- whether to rant or praise
- whether to be brief or detailed
- whether to sound analytical or expressive

So the BUILD for this phase will comprise:

- review tone mapping
- emotional intensity
- verbosity
- narrative structure
- criticism style
- praise emphasis
- Nigerian conversational flavor

In [56]:
# STEP 29 — CREATE TONE MAPPING
# this maps the predicted star ratings to a tone that could be used in a review generation 
# system to create more personalized and context-aware reviews based on the predicted sentiment 
# of the user towards the business.
tone_mapping = {

    5: "enthusiastic",

    4: "positive",

    3: "balanced",

    2: "disappointed",

    1: "frustrated"
}

In [57]:
# STEP 30 — TEST TONE MAPPING. CREATE EMOTIONAL INTENSITY MAPPING
# for rating in range(1, 6):

    # tone = tone_mapping[rating]

    # print(f"{rating} stars → {tone}")

archetype_intensity = {

    "Warm Optimist": "medium",

    "Reactive Reviewer": "high",

    "Harsh Critic": "high",

    "Emotional Storyteller": "very_high",

    "Deep Experience Analyst": "low"
}

In [58]:
# STEP 31 — CREATE VERBOSITY MAPPING
# This maps the archetypes to a verbosity level that could be used in a review generation 
# system to create more personalized and archetype-consistent reviews based on the user's communication style.

verbosity_mapping = {

    "Warm Optimist": "medium",

    "Reactive Reviewer": "short",

    "Harsh Critic": "medium",

    "Emotional Storyteller": "long",

    "Deep Experience Analyst": "very_long"
}

In [59]:
# STEP 32 — CREATE CRITICISM STYLE MAPPING
# This maps the archetypes to a criticism style that could be used in a review generation system 
# to create more personalized and archetype-consistent reviews based on the user's communication style.

criticism_style_mapping = {

    "Warm Optimist":
        "forgiving",

    "Reactive Reviewer":
        "emotionally_reactive",

    "Harsh Critic":
        "direct",

    "Emotional Storyteller":
        "dramatic",

    "Deep Experience Analyst":
        "analytical"
}

In [60]:
# STEP 33 — CREATE NIGERIAN IDENTITY STYLE MAPPING
# This maps the Nigerian identities to a style that could be used in a review generation system 
# to create more personalized and culturally resonant reviews based on the user's specific Nigerian identity.

nigerian_flavor_mapping = {

    "soft_life":
        "luxury_lagos_style",

    "social_vibes":
        "expressive_lagos_style",

    "value_sensitive":
        "practical_nigerian_style",

    "hustle_minded":
        "hustle_nigerian_style",

    "sarcastic_dramatic":
        "sarcastic_nigerian_style",

    "neutral":
        "generic_nigerian_style"
}

In [61]:
nigerian_persona_map = {

    "Warm Optimist":
        "social_vibes",

    "Reactive Reviewer":
        "value_sensitive",

    "Harsh Critic":
        "value_sensitive",

    "Emotional Storyteller":
        "soft_life",

    "Deep Experience Analyst":
        "soft_life"
}

persona_df["nigerian_style"] = (
    persona_df["archetype"]
    .map(nigerian_persona_map)
)

In [62]:
# STEP 34 — BUILD REVIEW PLAN
# Now we can create a function that takes the persona information, the predicted rating, 
# and other relevant details to build a review generation plan that includes the tone, 
# emotional intensity, verbosity, criticism style, and Nigerian flavor that should be used 
# when generating a review for this user-business interaction.

def build_review_plan(
    persona_row,
    predicted_rating
):

    archetype = persona_row["archetype"]

    nigerian_style = (
        persona_row["nigerian_style"]
    )

    review_plan = {

        "tone":
            tone_mapping[predicted_rating],

        "emotional_intensity":
            archetype_intensity[archetype],

        "verbosity":
            verbosity_mapping[archetype],

        "criticism_style":
            criticism_style_mapping[archetype],

        "nigerian_flavor":
            nigerian_flavor_mapping[
                nigerian_style
            ],

        "predicted_rating":
            predicted_rating
    }

    return review_plan

In [63]:
# STEP 35 — TEST REVIEW PLAN BUILDING
# Now we can see how the review plan is built based on the persona's archetype and the 
# predicted rating, which will guide the review generation system in creating a personalized 
# and context-aware review.

sample_persona = persona_df.iloc[4]

predicted_rating = 4

build_review_plan (
    sample_persona,
    predicted_rating
)

{'tone': 'positive',
 'emotional_intensity': 'high',
 'verbosity': 'short',
 'criticism_style': 'emotionally_reactive',
 'nigerian_flavor': 'practical_nigerian_style',
 'predicted_rating': 4}

In [81]:
# STEP 36 — CREATE REVIEW OPENING TEMPLATES
# Finally, we can create a set of opening sentence templates for reviews that correspond to different tones and archetypes. 
# These templates can be used by the review generation system to create more personalized and archetype-consistent reviews based on the user's predicted sentiment and communication style.

opening_styles = {

    "enthusiastic": [
        "Absolutely loved this place.",
        "This spot exceeded expectations.",
        "One of the best experiences I've had."
        "Really enjoyed my visit here.",
        "Solid experience overall.",
        "Had a good time here."
    ],

    "excited_positive": [
        "I'm still smiling as I write this.",
        "Chai! This place surprised me in the best way.",
        "Where do I even start? Absolute gem!",
        "See glass! This one na correct spot.",
        "If you no try this place, you're missing o.",
        "I dey feel good just remembering this experience.",
        "Omo, this place is a vibe from start to finish.",
        "Finally, somewhere that gets it right."
        "I am giving 5 stars because there is no room for 6"
    ],

    "positive": [
        "Really enjoyed my visit here.",
        "Solid experience overall.",
        "Had a good time here."
    ],
    
    "satisfied_content": [
        "No wahala experience from start to end.",
        "I genuinely enjoyed my time here.",
        "Solid 4 stars – no complaints, just small observations.",
        "It does what it says on the tin. Reliable.",
        "You know that feeling when everything just works? That was here.",
        "Nothing too fancy, but very solid.",
        "I would happily come back anytime."
    ],
    
    "balanced": [
        "Mixed feelings about this place.",
        "Some things worked, others didn't.",
        "Decent experience overall."
    ],

    "disappointed": [
        "I regret stepping foot in this place.",
        "Nawa o. Where do I even start?",
        "This one hurt my pocket and my spirit.",
        "See disappointment. Chai!",
        "I asked myself 'why did I come here?' the whole time.",
        "Never again. I mean it.",
        "The only thing worse than the food was the service.",
        "Story for the gods – and not the good kind."
        "Expected much better honestly.",
        "Left somewhat disappointed.",
        "The experience was underwhelming."
    ],
    
    "frustrated": [
        "I am fuming as I type this.",
        "This is the worst customer service I have ever received.",
        "See wahala! They have nerve o.",
        "I want my money back and my time back.",
        "Who send me? Abeg, avoid this place.",
        "E shock me that this place is still in business.",
        "I shouted at them – and I never shout."
        "This experience was genuinely frustrating.",
        "Would not return after this visit.",
        "Very disappointing experience."
    ],
    
    "sarcastic_playful": [
        "Oh wow. Just wow. (Sarcasm fully intended).",
        "Congratulations to them for the audacity.",
        "If zero stars was possible, I would give it.",
        "The only good thing was leaving.",
        "I’m not sure if the cook was angry at me personally.",
        "This place is… an experience. Not a good one.",
        "They tried. They failed. But they tried."
    ],
    
    "surprised_mixed": [
        "I didn't expect much, but wow – I was wrong.",
        "Honestly, I'm confused about how to rate this.",
        "It had good parts and bad parts. Let me explain.",
        "First half was terrible. Second half was amazing.",
        "My feelings are still conflicted.",
        "I wanted to love it, but..."
    ],
    
    "neutral_factual": [
        "Just the facts: here is my experience.",
        "No hype, no hate – just an honest review.",
        "I'll keep it short and straightforward.",
        "Let me break it down without sugarcoating.",
        "Here is what worked and what didn't."
    ],
    
    "family_oriented": [
        "Took the whole family there – here's how it went.",
        "My kids loved it, but my wallet didn't.",
        "I went with my people and we had a good time overall.",
        "This is a good spot for group outings.",
        "Even my uncle who complains about everything liked it."
    ],
    
    "romantic_date": [
        "Perfect date night spot – trust me.",
        "Took my babe here and the vibes were right.",
        "If you want to impress your partner, bring them here.",
        "Romantic, quiet, and classy. 10/10 for couples.",
        "The ambience alone is worth the visit."
    ],
    
    "hustle_budget": [
        "On a budget? This place won't kill your wallet.",
        "I went during sapa period and still managed to enjoy.",
        "Value for money? Yes. Luxury? No. Fair trade.",
        "Cheap and cheerful – nothing more, nothing less.",
        "My money didn't cry after leaving here."
    ],
    
    "hangry_urgent": [
        "I was starving when I arrived, so maybe I'm biased.",
        "They fed a hungry person – that counts for something.",
        "I almost lost my cool waiting, but the food saved them.",
        "If you're very hungry, this place will do.",
        "Fast service for a starving customer – thank you."
    ],
    
    "after_work_tired": [
        "Came here after a long day – just wanted to relax.",
        "I was too tired to complain, but honestly it was fine.",
        "Decent place to unwind after work.",
        "Low energy, high patience – and they delivered."
    ]
}

In [65]:
# STEP 37 — CREATE FOCUS AREA GENERATOR 
# Now we can see how the review opening templates correspond to different tones, focus, and 
# archetypes, and we can create a function that generates a focus area for the review 
# based on the user's dominant value, which will guide the content of the review to 
# align with what matters most to the user.

def determine_review_focus(
    persona_row
):

    dominant_value = (
        persona_row["dominant_value"]
    )

    if dominant_value == "ambience":

        return [
            "atmosphere", "aesthetics", "vibes", "lighting", "music", 
            "decor", "seating comfort", "view"
        ]

    elif dominant_value == "food_quality":

        return [
            "taste", "food quality", "portion sizes"
        ]

    elif dominant_value == "social_experience":

        return [
            "social energy", "music", "crowd", "energy", "DJ quality", "crowd liveliness", "drink selection",
            "dance floor", "after-hours", "security", "nightlife", "party atmosphere"
        ]

    elif dominant_value == "affordability":

        return [
            "pricing", "value", "price", "value for money", "cost", "budget-friendliness",
            "portion size vs price", "hidden charges", "discounts"
        ]

    elif dominant_value == "durability":

        return [
            "build quality", "longevity", "ruggedness", "original vs fake",
            "materials", "resistance to wear", "repairability"
        ]

    elif dominant_value == "service_quality":

        return [
            "staff attitude", "speed of service", "politeness", "attentiveness",
            "problem resolution", "wait time", "follow-up"
        ]

    elif dominant_value == "social_proof":

        return [
            "crowd popularity", "friend recommendations", "family approval",
            "neighbour's experience", "busyness", "word-of-mouth reputation"
        ]

    elif dominant_value == "time_efficiency":

        return [
            "waiting time", "delivery speed", "queue length", "preparation time",
            "punctuality", "response time"
        ]

    elif dominant_value == "luxury":

        return [
            "premium experience", "exclusivity", "high-end finishes", "brand prestige",
            "attentive VIP treatment", "aesthetics", "expensive but worth"
        ]
    
    elif dominant_value == "romance":

        return [
            "intimacy", "candlelight", "quiet corners", "couple seating",
            "date night suitability", "rosy atmosphere"
        ]

    elif dominant_value == "family_friendliness":

        return [
            "kid-friendly", "space for groups", "family seating", "child menu",
            "play area", "stroller access"
        ]

    elif dominant_value == "authenticity":

        return [
            "traditional recipes", "original preparation", "cultural accuracy",
            "local ingredients", "home-style cooking", "no shortcuts"
        ]


    return ["overall experience"]

In [66]:
determine_review_focus(
    sample_persona
)

['overall experience']

In [67]:
# STEP 38 — GENERATE REVIEW BLUEPRINT
# Finally, we can create a function that takes all the previous components — 
# the persona information, the predicted rating, the review plan, and the focus areas — 
# to generate a comprehensive blueprint for how a review should be generated for this 
# user-business interaction, which can then be used by a review generation system to 
# create a personalized, context-aware, and archetype-consistent review.

def generate_review_blueprint(
    persona_row,
    predicted_rating
):

    plan = build_review_plan(
        persona_row,
        predicted_rating
    )

    focus_areas = determine_review_focus(
        persona_row
    )

    blueprint = {

        "review_plan": plan,

        "focus_areas": focus_areas,

        "opening_examples":
            opening_styles[
                plan["tone"]
            ]
    }

    return blueprint

In [68]:
generate_review_blueprint(
    sample_persona,
    predicted_rating
)

{'review_plan': {'tone': 'positive',
  'emotional_intensity': 'high',
  'verbosity': 'short',
  'criticism_style': 'emotionally_reactive',
  'nigerian_flavor': 'practical_nigerian_style',
  'predicted_rating': 4},
 'focus_areas': ['overall experience'],
 'opening_examples': ['Really enjoyed my visit here.',
  'Solid experience overall.',
  'Had a good time here.']}

PHASE 4 — Review Planning Engine is now complete.
The system now has:

- Human personas	
- Value systems	
- Emotional drift	
- Nigerian contextualization
- Compatibility reasoning	
- Contextual reasoning	
- Rating prediction	
- Review planning

In [69]:
# PHASE 5 — ACTUAL REVIEW GENERATION

In [ ]:
# STEP 39 — BUILD REVIEW PROMPT
# Now we can create a function that takes the persona information, the business information, 
# the context, the predicted rating, and the review blueprint to build a detailed prompt that can
# be fed into a review generation system (like a language model) to produce a personalized and context-aware 
# review that aligns with the user's archetype, values, and current situation.

def build_review_prompt(
    persona_row,
    business_row,
    context_name,
    predicted_rating,
    review_blueprint
):

    prompt = f"""

You are simulating a realistic human Yelp reviewer.

USER PROFILE:
- Archetype: {persona_row['archetype']}
- Dominant Value: {persona_row['dominant_value']}
- Nigerian Identity: {persona_row['nigerian_identity']}
- Nigerian Communication Style: {persona_row['nigerian_style']}

CURRENT CONTEXT:
- Situation: {context_name}

RESTAURANT:
- Name: {business_row['name']}
- Categories: {business_row['categories']}
- Average Yelp Rating: {business_row['stars']}

PREDICTED USER EXPERIENCE:
- Expected Rating: {predicted_rating} stars

REVIEW STYLE PLAN:
- Tone: {review_blueprint['review_plan']['tone']}
- Emotional Intensity:
  {review_blueprint['review_plan']['emotional_intensity']}
- Verbosity:
  {review_blueprint['review_plan']['verbosity']}
- Criticism Style:
  {review_blueprint['review_plan']['criticism_style']}
- Nigerian Flavor:
  {review_blueprint['review_plan']['nigerian_flavor']}

FOCUS AREAS:
{', '.join(review_blueprint['focus_areas'])}


INSTRUCTIONS:
- Write EXACTLY like a real human reviewer.
- Avoid sounding like AI.
- Be conversational and emotionally natural.
- Use realistic restaurant details.
- Match the user's personality strongly.
- Match the predicted emotional tone.
- Use subtle Nigerian conversational style where appropriate.
- Sound culturally Nigerian where appropriate.
- Do NOT explain yourself.
- Do NOT summarize.
- Do NOT use bullet points.
- Do NOT overuse slang.
- Write ONLY the review.
- Sound spontaneous.
- Include imperfections in expression naturally.

"""

    return prompt

In [86]:
# STEP 40 — TEST REVIEW PROMPT BUILDING
# Now we can see how the review prompt is built based on all the previous components, 
# which will guide the review generation system in creating a personalized and context-aware review.

sample_persona = persona_df.iloc[5]

sample_business = business_profiles.iloc[53]

context_name = "celebration"

compatibility_result = (
    contextualized_compatibility(
        sample_persona,
        sample_business,
        context_name
    )
)

predicted_rating = (
    contextual_rating_prediction(
        sample_persona,
        compatibility_result,
        context_name
    )
)

review_blueprint = (
    generate_review_blueprint(
        sample_persona,
        predicted_rating
    )
)

prompt = build_review_prompt(
    sample_persona,
    sample_business,
    context_name,
    predicted_rating,
    review_blueprint
)

print(prompt)



You are simulating a realistic human Yelp reviewer.

USER PROFILE:
- Archetype: Warm Optimist
- Dominant Value: service_quality
- Nigerian Identity: pidgin_staples
- Nigerian Communication Style: social_vibes

CURRENT CONTEXT:
- Situation: celebration

RESTAURANT:
- Name: Paws The Cat Cafe
- Categories: Coffee & Tea, Cafes, Pets, Restaurants, Pet Adoption, Food
- Average Yelp Rating: 5.0

PREDICTED USER EXPERIENCE:
- Expected Rating: 4 stars

REVIEW STYLE PLAN:
- Tone: positive
- Emotional Intensity:
  medium
- Verbosity:
  medium
- Criticism Style:
  forgiving
- Nigerian Flavor:
  expressive_lagos_style

FOCUS AREAS:
staff attitude, speed of service, politeness, attentiveness, problem resolution, wait time, follow-up

INSTRUCTIONS:
- Write like a real human.
- Sound culturally Nigerian where appropriate.
- Be emotionally believable.
- Mention realistic details.
- Avoid sounding robotic.
- Do NOT overuse slang.
- Output ONLY the review text.




In [87]:
# STEP 41 — GENERATE REVIEW USING LANGUAGE MODEL
# Finally, we can take the generated prompt and feed it into a language model (like GPT-4) 
# to produce a personalized, context-aware, and archetype-consistent review that simulates a 
# realistic human review based on the user's profile, the business details, and the current situation.

def generate_review(prompt):

    response = model.generate_content(
        prompt
    )

    return response.text

In [95]:
# STEP 42 — TEST REVIEW GENERATION

generated_review = generate_review(prompt)

print(generated_review)

Omo, this place is just pure joy and vibes! We came here to celebrate my sister’s promotion—just a sweet, happy weekend treat—and Paws The Cat Cafe really made our day. 

The moment we walked in, the staff welcomed us with the biggest, warmest smiles. Honestly, their customer service is top-notch. They were so polite and attentive, explaining how to play with the cats so we wouldn't stress the poor babies, and making sure we settled in well. You can tell they actually love what they do.

My only tiny comment was the wait time. It took a bit of time for our lattes and red velvet cake to arrive because the cafe was quite busy. But the way the staff handled it? I was so impressed. The lady serving us came over, apologized so nicely, and kept us updated. You can't even vex when someone is that sweet and professional about their work. 

The cats themselves are so clean and beautiful, just roaming around and bringing calm. We had such a lovely time celebrating. It's a solid 4 stars from me, 

In [96]:
def simulate_review(
    persona_row,
    business_row,
    context_name
):

    compatibility_result = (
        contextualized_compatibility(
            persona_row,
            business_row,
            context_name
        )
    )

    predicted_rating = (
        contextual_rating_prediction(
            persona_row,
            compatibility_result,
            context_name
        )
    )

    review_blueprint = (
        generate_review_blueprint(
            persona_row,
            predicted_rating
        )
    )

    prompt = build_review_prompt(
        persona_row,
        business_row,
        context_name,
        predicted_rating,
        review_blueprint
    )

    review_text = generate_review(prompt)

    return {

        "predicted_rating":
            predicted_rating,

        "review_text":
            review_text,

        "review_blueprint":
            review_blueprint,

        "compatibility_result":
            compatibility_result
    }

In [97]:
# STEP 43 — SIMULATE FULL REVIEW GENERATION PIPELINE
# Now we can simulate the entire pipeline from persona selection to review generation for 
# multiple iterations to see how different personas, businesses, and contexts lead to different 
# predicted ratings and review outputs.
import random

contexts = [
    "celebration",
    "social_night_out",
    "weekday_quick_meal",
    "comfort_food_mood"
]

for i in range(10):

    persona = persona_df.sample(1).iloc[0]

    business = business_profiles.sample(1).iloc[0]

    context = random.choice(contexts)

    result = simulate_review(
        persona,
        business,
        context
    )

    print("=" * 70)

    print("ARCHETYPE:",
          persona["archetype"])

    print("CONTEXT:",
          context)

    print("RATING:",
          result["predicted_rating"])

    print("\n")

    print(result["review_text"])

    print("\n")

ARCHETYPE: Emotional Storyteller
CONTEXT: comfort_food_mood
RATING: 2


I am actually sitting in my car writing this, still trembling with disappointment because I cannot believe how my afternoon was ruined. My alternator was acting up, and since I’m a firm believer in the soft life—I cannot come and kill myself because of car stress—I decided to wait at Heng’s. Someone had told me they have a little canteen corner inside the waiting area that serves "amazing comfort food" while you wait. 

Abeg, who lied to them? 

I was already in a mood, just craving something warm, rich, and comforting to soothe my nerves. I ordered their beef broth and noodle bowl. When the plate arrived, my jaw literally dropped. The portion size? It was an insult. An absolute joke. Is this food for a toddler or a grown adult paying good money? It was barely three forkfuls of noodles swimming in a sad, shallow puddle. 

But okay, I said let me chop first, maybe the taste will make up for the size. 

My people, it